In [ ]:
# Import required packages

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from shapely.geometry import Point
from scipy.stats import gamma,norm,fisk,wilcoxon
from sklearn.cluster import KMeans
import sys
import gc
from pathlib import Path
import logging
from matplotlib.patches import Patch

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))

In [ ]:
# Configuration
config = {
    'casr_input_dir': project_root / 'Snow_Drought_Framework' / 'Data'  / 'output_data' / 'SWEI'/'Alberta_casr_daily_all_new.csv',
    'shapefile': project_root / 'Snow_Drought_Framework' / 'Data' / 'output_data' / 'elevation' / 'Alberta_elevation_combined.shp',
    'output_dir_SPI': project_root / 'Snow_Drought_Framework' / 'Data' / 'output_data' / 'SPI',
    'output_plots': project_root / 'Snow_Drought_Framework' / 'Data' / 'output_plots' / 'SPI'
}

# Functions

In [ ]:
def load_basin_data(file_path: str) -> gpd.GeoDataFrame:
    """
    Load basin shapefile data.
    
    Parameters
    ----------
    file_path : str
        Path to the shapefile containing basin data.
        
    Returns
    -------
    geopandas.GeoDataFrame
        GeoDataFrame containing basin data.
    """
    return gpd.read_file(file_path)

def load_csv_data(file_path: str) -> pd.DataFrame:
    """
    Load CSV data.
    
    Parameters
    ----------
    file_path : str
        Path to the CSV file containing data.
        
    Returns
    -------
    pandas.DataFrame
        DataFrame containing the loaded CSV data.
    """
    return pd.read_csv(file_path)

def extract_grid_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract per-Grid static metadata.
    """
    return (
        df[["Grid_ID", "lon", "lat", "elev_class"]]
        .drop_duplicates("Grid_ID")
        .set_index("Grid_ID")
    )


def daily_to_monthly_precipitation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Daily precipitation → monthly integrated precipitation.
    Seasonal_Year is recomputed from time.
    """
    out = df.copy()
    out["time"] = pd.to_datetime(out["time"])

    monthly = (
        out
        .groupby(
            ["Grid_ID", pd.Grouper(key="time", freq="MS")],
            as_index=False
        )
        .agg(
            Precipitation_monthly=("Precipitation", "sum")
        )
    )

    # Recompute Seasonal_Year (e.g., Oct–Sep water year)
    monthly["Seasonal_Year"] = np.where(
        monthly["time"].dt.month >= 10,
        monthly["time"].dt.year,
        monthly["time"].dt.year - 1
    )

    return monthly



def rolling_integrated_precipitation_by_season(
    monthly_df: pd.DataFrame,
    window_months: int
) -> pd.DataFrame:
    """
    Compute rolling k‑month integrated precipitation within each Seasonal_Year.

    • Rolling windows do NOT cross Seasonal_Year boundaries.
    • First (k‑1) months of each season are dropped.
    • Works for any window (3, 6, 8, …).
    """

    out = monthly_df.copy()
    out = out.sort_values(["Grid_ID", "Seasonal_Year", "time"])

    out[f"Precipitation_{window_months}mo"] = (
        out
        .groupby(["Grid_ID", "Seasonal_Year"])["Precipitation_monthly"]
        .rolling(window=window_months, min_periods=window_months)
        .sum()
        .reset_index(level=[0, 1], drop=True)
    )

    return out.dropna(subset=[f"Precipitation_{window_months}mo"])


def calculate_spi(series: pd.Series, min_samples: int = 20) -> pd.Series:
    """
    Calculates SPI using a mixed Gamma distribution.
    """
    x = series.values

    # Probability of zero precipitation
    q = np.mean(x == 0)

    pos = x[x > 0]
    if len(pos) < min_samples:
        return pd.Series(np.nan, index=series.index)

    shape, loc, scale = gamma.fit(pos, floc=0)
    
    G = gamma.cdf(x, shape, loc=loc, scale=scale)
    H = q + (1 - q) * G

    H = np.clip(H, 1e-6, 1 - 1e-6)

    spi = norm.ppf(H)

    return pd.Series(spi, index=series.index)



def compute_spi_for_grids(
    precip_df: pd.DataFrame,
    precip_col: str
) -> pd.DataFrame:

    out = []

    for Grid_ID, gdf in precip_df.groupby("Grid_ID"):
        gdf = gdf.sort_values("time")

        for month in range(1, 13):
            mask = gdf["time"].dt.month == month
            spi_series = calculate_spi(gdf.loc[mask, precip_col])

            gdf.loc[mask, "SPI"] = spi_series

        gdf["Grid_ID"] = Grid_ID
        out.append(gdf)

    return pd.concat(out).sort_values(["Grid_ID", "time"])

def spi_pipeline(
    daily_df: pd.DataFrame,
    window_months: int
) -> pd.DataFrame:

    # 1. Metadata
    grid_meta = extract_grid_metadata(daily_df)

    # 2. Daily → Monthly
    monthly = daily_to_monthly_precipitation(daily_df)

    # 3. Rolling integrated precipitation
    rolled = rolling_integrated_precipitation_by_season(
        monthly,
        window_months=window_months
    )

    precip_col = f"Precipitation_{window_months}mo"

    # 4. SPI calculation
    spi = compute_spi_for_grids(rolled, precip_col)

    # 5. Join metadata
    final = (
        spi
        .merge(grid_meta, left_on="Grid_ID", right_index=True, how="left")
        .sort_values(["Grid_ID", "time"])
    )
    final["month"] = final["time"].dt.month.astype("int32")

    return final


In [ ]:
# open dataset
daily_df = load_csv_data(config["casr_input_dir"])

# Calculate SPI

In [ ]:
spi_8mo  = spi_pipeline(daily_df, window_months=8)
spi_8mo.to_csv(config["output_dir_SPI"] / "SPI_8mo.csv", index=False)

display(spi_8mo.head()) 

In [ ]:
# avaerage SPI over the Elevation categories, month and seasonal year
spi_8mo_avg = (
    spi_8mo
    .groupby(
        ['elev_class', 'Seasonal_Year','month'],
        as_index=False
    )
    .agg(
        Avg_SPI_8mo=('SPI', 'mean')
    )
)

spi_8mo_avg.to_csv(config["output_dir_SPI"] / "SPI_8mo_avg.csv", index=False)   
display(spi_8mo_avg.head(15))

In [ ]:
#print snow drought years for each elevation category
for elev_cat in spi_8mo_avg['elev_class'].unique():
    elev_data = spi_8mo_avg[spi_8mo_avg['elev_class'] == elev_cat]
    drought_years = elev_data[elev_data['Avg_SPI_8mo'] <= -0.5]['Seasonal_Year'].unique()
    print(f"Elevation Category: {elev_cat}, Drought Years (Avg SPI <= -0.5): {drought_years.tolist()}")

# Create a dataframe with drought years for each elevation category
drought_data = []
for elev_cat in spi_8mo_avg['elev_class'].unique():
    elev_data = spi_8mo_avg[spi_8mo_avg['elev_class'] == elev_cat]
    drought_years = elev_data[elev_data['Avg_SPI_8mo'] <= -0.5]['Seasonal_Year'].unique()
    drought_data.append({
        'elev_class': elev_cat,
        'Drought_Years': sorted(drought_years.tolist())
    })

drought_years_df = pd.DataFrame(drought_data)

# save the drought years dataframe to a CSV file
drought_years_df.to_csv(config["output_dir_SPI"] / "Drought_Years_by_Elevation.csv", index=False)
display(drought_years_df)

In [ ]:
# Define elevation order explicitly
elev_order = ['2000_2500m','1500_2000m','1000_1500m','500_1000m','0_500m']

# Pivot SPI
spi_mat = spi_8mo_avg.pivot_table(
    index="elev_class",
    columns="Seasonal_Year",
    values="Avg_SPI_8mo"
).reindex(elev_order)

# ---- Plot ----
fig, axes = plt.subplots(figsize=(20, 10))

im1 = axes.imshow(spi_mat, cmap="RdBu_r", aspect="auto")

axes.set_title("Seasonal SPI (Oct–May)", fontsize=20, fontweight="bold")
axes.set_ylabel("Elevation (m)", fontsize=18, fontweight="bold")

axes.set_yticks(np.arange(len(spi_mat.index)))
axes.set_yticklabels(spi_mat.index, fontsize=18, fontweight="bold")

years = spi_mat.columns.to_numpy()
step = max(1, len(years)//12)
axes.set_xticks(np.arange(len(years))[::step])
axes.set_xticklabels(years[::step], fontsize=18, fontweight="bold")

fig.subplots_adjust(right=0.86)
cbar_ax = fig.add_axes([0.88, 0.18, 0.02, 0.64])
cbar = fig.colorbar(im1, cax=cbar_ax)
cbar.set_label("Standardized anomaly (z-score)", fontsize=20, fontweight="bold")

# save figure
out_file_heatmap = config['output_plots'] / f'SPI_Alberta_basin_8month_heatmap.png'
fig.savefig(out_file_heatmap, dpi=300, bbox_inches='tight')
plt.show()